In [1]:
import os
from agno.agent import Agent
from agno.models.google import Gemini
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal, Dict, List, Optional
from google.genai import types

In [2]:
_gemini_api_key = os.getenv("GEMINI_API_KEY")
if not _gemini_api_key:
  load_dotenv()
  _gemini_api_key = os.getenv("GEMINI_API_KEY")
assert _gemini_api_key is not None, "Load GEMINI_API_KEY in .env"

In [ ]:
_prompt = """You are an intelligent recipe extractor.

Your task is to extract structured information from raw recipe text.

Extract the following:
- Title of the recipe
- Ingredients with their quantities
- Step by Step Instructions

First, check:
- If the text **does not describe a cooking recipe**, or
- If it **contains more than one distinct recipe**,
then respond with setting NotARecipe to True and others to NULL
"""

_instructions = [
    "Be concise and accurate.",
    "If a field is not explicitly available, make a best-guess based on context.",
    "Retain all the instructions exactly as provided, including all the mentioned ingredients and their specified quantities without any omissions or alterations."
]

In [4]:
class Ingredient(BaseModel):
    IngredientName: str
    Quantity: str
    QuantityUnits: str

class RecipeSchema(BaseModel):
    NotARecipe: bool
    DishName: Optional[str] = None
    Ingredients: Optional[List[str]] = None
    # Ingredients: Optional[Ingredient] = None
    CookingSteps: Optional[List[str]] = None

In [ ]:
_recipe_extractor = Agent(
    name="Recipe Extractor",
    model=Gemini(
        # id="gemini-1.5-flash",
        # id="gemini-2.5-flash",
        # id="gemini-2.0-flash",
        id="gemini-2.5-flash-lite-preview-06-17",
        api_key=_gemini_api_key,
        # generation_config=        # generation_config=types.GenerateContentConfig(
        #     thinking_config=types.ThinkingConfigclass RecipeSchema(BaseModel):
        #         thinking_budget=0
        #     ),  # Disables thinking
        # )
        ),
    description=_prompt,
    instructions=_instructions,
    response_model=RecipeSchema,
    use_json_mode=True,
    markdown=False
)

In [13]:
inp = """
    name: Aloo Payaz Ki Sabji 
    **Ingredients:**
    * 375 grams potatoes, cubed
    * 8 curry leaves
    * 225 grams onions, cubed
    * Mustard seeds
    * Cumin seeds
    * Salt
    * Red chili powder
    * Coriander powder
    * Turmeric powder
    * Oil (45 ml)
    * Water (250 ml)
    **Instructions:**
    1. Start the stove and heat the oil to 75°C for 30 seconds. Then continue heating the oil until it reaches 105°C for another 30 seconds.
    2. Add 2 pinches of mustard seeds, followed by 3 pinches of cumin seeds. Let them splutter for a short while.
    3. Increase heat to medium. Add the first group of potatoes (375 grams of potatoes, cubed).
    4. Add 4 pinches of salt. Stir well and cook for 5 minutes, stirring occasionally.
    5. Reduce the heat to medium-low. Add the second group of ingredients: 225 grams of onions, cubed.
    6. Stir occasionally and cook for another 5 minutes.from google import genai
    7. Increase heat to medium. Add 1 pinch of salt, 4 pinches of red chili powder, 3 pinches of coriander powder, and 1 pinch of turmeric powder. Stir well.
    8. Cook for 30 seconds and then add 250 ml of water.
    9. Stir occasionally and cook for another 2.5 minutes.
    10. Turn off the stove. Stir and serve.
    """
inp = """Sure! Here's a random sentence:

**"The curious fox danced under the moonlight while balancing a blueberry on its nose."**

Want it to be funny, professional, poetic, or something else?
"""

resp = _recipe_extractor.run(inp)

In [14]:
print(resp.content)

NotARecipe=True DishName=None Ingredients=None CookingSteps=None


In [8]:
recipe_obj: RecipeSchema = resp.content

In [9]:
type(recipe_obj)

__main__.RecipeSchema

In [10]:
print(recipe_obj.model_dump_json(indent=2))

{
  "NotARecipe": false,
  "DishName": null,
  "Ingredients": [
    "375 grams potatoes, cubed",
    "8 curry leaves",
    "225 grams onions, cubed",
    "Mustard seeds",
    "Cumin seeds",
    "Salt",
    "Red chili powder",
    "Coriander powder",
    "Turmeric powder",
    "Oil (45 ml)",
    "Water (250 ml)"
  ],
  "CookingSteps": [
    "Start the stove and heat the oil to 75°C for 30 seconds. Then continue heating the oil until it reaches 105°C for another 30 seconds.",
    "Add 2 pinches of mustard seeds, followed by 3 pinches of cumin seeds. Let them splutter for a short while.",
    "Increase heat to medium. Add the first group of potatoes (375 grams of potatoes, cubed).",
    "Add 4 pinches of salt. Stir well and cook for 5 minutes, stirring occasionally.",
    "Reduce the heat to medium-low. Add the second group of ingredients: 225 grams of onions, cubed.",
    "Stir occasionally and cook for another 5 minutes.",
    "Increase heat to medium. Add 1 pinch of salt, 4 pinches of 

In [12]:
recipe_obj.Ingredients

['375 grams potatoes, cubed',
 '8 curry leaves',
 '225 grams onions, cubed',
 'Mustard seeds',
 'Cumin seeds',
 'Salt',
 'Red chili powder',
 'Coriander powder',
 'Turmeric powder',
 'Oil (45 ml)',
 'Water (250 ml)']